# SurfaceTarget-CRC — phase 01 — patch 1 9

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.9 (CORRECTED)
#
# MSS Evidence Tiers and Sample Identifier Validation
#
# Objectives:
# 1. Define preliminary MSS evidence tiers
# 2. Validate GDC / cBioPortal sample correspondence
# 3. Exclude published MSI-L from strict MSS eligibility
# 4. Flag sample mismatches for review
# 5. Save reproducible patient-level cohort metadata
#
# IMPORTANT:
# This is a cohort eligibility audit.
# Final RNA-seq sample selection has not been performed.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import json
import re

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

COMPARISON_PATH = (
    METADATA_DIR /
    "tcga_computational_vs_published_msi.csv"
)

PUBLISHED_PATH = (
    METADATA_DIR /
    "tcga_published_msi_status_raw.csv"
)

INVENTORY_PATH = (
    METADATA_DIR /
    "tcga_rnaseq_file_inventory.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.9 — MSS COHORT EVIDENCE VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load existing data
# ------------------------------------------------------------

required_files = [
    COMPARISON_PATH,
    PUBLISHED_PATH,
    INVENTORY_PATH
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

comparison = pd.read_csv(
    COMPARISON_PATH,
    dtype={"patientId": str}
)

published = pd.read_csv(
    PUBLISHED_PATH,
    dtype=str
)

inventory = pd.read_csv(
    INVENTORY_PATH,
    dtype=str
)

print("\nData loaded successfully.")

print("Patient-level records:", len(comparison))

print("Published MSI records:", len(published))

print(
    "GDC RNA-seq files:",
    inventory["file_id"].nunique()
)

# ------------------------------------------------------------
# 3. Validate patient identifiers
# ------------------------------------------------------------

if comparison["patientId"].duplicated().any():
    raise RuntimeError(
        "Duplicate patient identifiers detected."
    )

required_groups = {
    "Concordant low MSI",
    "Concordant MSI-H",
    "Discordant",
    "Incomplete"
}

observed_groups = set(
    comparison["msi_group"].dropna()
)

if not observed_groups.issubset(required_groups):
    raise RuntimeError(
        "Unexpected computational MSI classifications."
    )

print("\nPatient identifier validation: PASSED")

# ------------------------------------------------------------
# 4. Normalize published classifications
# ------------------------------------------------------------

published["status_clean"] = (
    published["status_clean"]
    .astype("string")
    .str.strip()
    .str.upper()
)

allowed_statuses = {
    "MSS",
    "MSI-L",
    "MSI-H",
    "NOT EVALUABLE"
}

unexpected = (
    set(published["status_clean"].dropna())
    - allowed_statuses
)

if unexpected:
    raise RuntimeError(
        f"Unexpected published MSI labels: {unexpected}"
    )

# Require one published annotation per patient.

if published["patientId"].duplicated().any():
    raise RuntimeError(
        "Multiple published annotations per patient. "
        "Manual review is required."
    )

published_subset = published[
    ["patientId", "sampleId", "status_clean"]
].rename(
    columns={
        "sampleId": "published_sample_id",
        "status_clean": "published_msi_status"
    }
)

# ------------------------------------------------------------
# 5. Merge published annotations
# ------------------------------------------------------------

cohort = comparison.drop(
    columns=["published_status"],
    errors="ignore"
).merge(
    published_subset,
    on="patientId",
    how="left",
    validate="one_to_one"
)

print("\nPublished annotation distribution:")

print(
    cohort["published_msi_status"]
    .fillna("NOT AVAILABLE")
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 6. Prepare primary tumor RNA-seq samples
# ------------------------------------------------------------

primary_inventory = inventory[
    inventory["sample_type"] == "Primary Tumor"
].copy()

primary_inventory = primary_inventory[
    [
        "patient_id",
        "sample_id",
        "project",
        "file_id"
    ]
].drop_duplicates()

gdc_samples = (
    primary_inventory
    .groupby("patient_id")["sample_id"]
    .apply(
        lambda values: set(values.dropna())
    )
    .to_dict()
)

print("\nPrimary tumor sample inventory prepared.")

# ------------------------------------------------------------
# 7. TCGA sample barcode normalization
# ------------------------------------------------------------

def tcga_sample_key(sample_id):
    """
    Extract TCGA barcode through sample-type code.

    Example:
    TCGA-XX-XXXX-01A -> TCGA-XX-XXXX-01

    This identifies the same patient and sample-type
    category, but does NOT prove identical aliquots.
    """

    if pd.isna(sample_id):
        return None

    value = str(sample_id).strip().upper()

    match = re.match(
        r"^(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-[0-9]{2})",
        value
    )

    return match.group(1) if match else None


def check_gdc_sample(patient_id, sample_id):
    """
    Compare a cBioPortal sample identifier with
    primary tumor RNA-seq samples from GDC.
    """

    if pd.isna(sample_id):
        return "Not available"

    reference_samples = gdc_samples.get(
        patient_id,
        set()
    )

    if not reference_samples:
        return "No GDC primary tumor sample"

    sample_id = str(sample_id).strip().upper()

    normalized_references = {
        str(s).strip().upper()
        for s in reference_samples
    }

    if sample_id in normalized_references:
        return "Exact sample ID match"

    query_key = tcga_sample_key(sample_id)

    reference_keys = {
        tcga_sample_key(s)
        for s in reference_samples
    }

    if (
        query_key is not None
        and query_key in reference_keys
    ):
        return "Sample-type barcode match"

    return "No matching GDC sample"

# ------------------------------------------------------------
# 8. Validate PanCancer Atlas sample IDs
# ------------------------------------------------------------

cohort["pancancer_gdc_match"] = cohort.apply(
    lambda row: check_gdc_sample(
        row["patientId"],
        row["mantis_sample_id"]
    ),
    axis=1
)

# ------------------------------------------------------------
# 9. Validate published MSI sample IDs
# ------------------------------------------------------------

cohort["published_gdc_match"] = cohort.apply(
    lambda row: check_gdc_sample(
        row["patientId"],
        row["published_sample_id"]
    ),
    axis=1
)

# ------------------------------------------------------------
# 10. Compare cBioPortal sample identifiers
# ------------------------------------------------------------

cohort["cbio_sample_match"] = (
    cohort["mantis_sample_id"].notna()
    & cohort["published_sample_id"].notna()
    & (
        cohort["mantis_sample_id"]
        .astype("string")
        .str.strip()
        .str.upper()
        ==
        cohort["published_sample_id"]
        .astype("string")
        .str.strip()
        .str.upper()
    )
).fillna(False).astype(bool)

# ------------------------------------------------------------
# 11. Report sample correspondence
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAMPLE CORRESPONDENCE AUDIT")
print("-" * 65)

print("\nPanCancer Atlas vs GDC:")

print(
    cohort["pancancer_gdc_match"]
    .value_counts()
    .to_string()
)

print("\nPublished study vs GDC:")

print(
    cohort["published_gdc_match"]
    .value_counts()
    .to_string()
)

print("\nExact matches between cBioPortal studies:")

print(
    cohort["cbio_sample_match"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 12. Construct preliminary evidence tiers
#
# FIX:
# Explicitly convert all classification conditions
# to standard Boolean arrays without missing values.
# ------------------------------------------------------------

low_msi = (
    cohort["msi_group"]
    .eq("Concordant low MSI")
    .fillna(False)
    .astype(bool)
)

published_mss = (
    cohort["published_msi_status"]
    .eq("MSS")
    .fillna(False)
    .astype(bool)
)

published_msil = (
    cohort["published_msi_status"]
    .eq("MSI-L")
    .fillna(False)
    .astype(bool)
)

published_missing = (
    cohort["published_msi_status"]
    .isna()
    .astype(bool)
)

valid_match_types = {
    "Exact sample ID match",
    "Sample-type barcode match"
}

pancancer_linked = (
    cohort["pancancer_gdc_match"]
    .isin(valid_match_types)
    .fillna(False)
    .astype(bool)
)

published_linked = (
    cohort["published_gdc_match"]
    .isin(valid_match_types)
    .fillna(False)
    .astype(bool)
)

cbio_linked = (
    cohort["cbio_sample_match"]
    .fillna(False)
    .astype(bool)
)

# ------------------------------------------------------------
# 13. Define Tier A
# ------------------------------------------------------------

# Tier A:
# Published MSS + concordant computational low MSI
# + cross-study sample correspondence.

tier_a = (
    low_msi
    & published_mss
    & pancancer_linked
    & published_linked
    & cbio_linked
)

# ------------------------------------------------------------
# 14. Define Tier B
# ------------------------------------------------------------

# Tier B:
# Concordant computational low MSI
# without a published MSI classification.

tier_b = (
    low_msi
    & published_missing
    & pancancer_linked
)

# ------------------------------------------------------------
# 15. Assign explicit evidence categories
#
# FIX:
# Convert every condition to a NumPy Boolean array.
# This prevents the previous np.select TypeError.
# ------------------------------------------------------------

conditions = [
    tier_a,
    tier_b,
    low_msi & published_msil,
    low_msi & published_mss & ~tier_a,
    low_msi & published_missing & ~tier_b
]

conditions = [
    condition.fillna(False).to_numpy(dtype=bool)
    for condition in conditions
]

choices = [
    "Tier A - Published MSS and concordant low MSI",
    "Tier B - Concordant low MSI only",
    "Excluded - Published MSI-L",
    "Review - MSS sample linkage",
    "Review - Low-MSI sample linkage"
]

cohort["evidence_tier"] = np.select(
    conditions,
    choices,
    default="Other - Not eligible for primary MSS cohort"
)

# Validate that every patient has a category.

if cohort["evidence_tier"].isna().any():
    raise RuntimeError(
        "Some patients have missing evidence tiers."
    )

print("\nEvidence tier assignment: PASSED")

# ------------------------------------------------------------
# 16. Display evidence tier results
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSS EVIDENCE TIER DISTRIBUTION")
print("-" * 65)

print("\nOverall:")

print(
    cohort["evidence_tier"]
    .value_counts()
    .to_string()
)

print("\nBy TCGA project:")

tier_summary = pd.crosstab(
    cohort["gdc_project"],
    cohort["evidence_tier"]
)

print(tier_summary.to_string())

# ------------------------------------------------------------
# 17. Extract eligible cohorts
# ------------------------------------------------------------

strict_mss = cohort[
    tier_a
].copy()

extended_low_msi = cohort[
    tier_b
].copy()

print("\n" + "-" * 65)
print("ELIGIBLE RESEARCH COHORTS")
print("-" * 65)

print("\nTier A - Published MSS + concordant low MSI")

print("Total:", len(strict_mss))

print(
    strict_mss["gdc_project"]
    .value_counts()
    .to_string()
)

print("\nTier B - Computationally low MSI only")

print("Total:", len(extended_low_msi))

print(
    extended_low_msi["gdc_project"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 18. Identify discrepancies for review
# ------------------------------------------------------------

review = cohort[
    cohort["evidence_tier"]
    .str.startswith("Review")
].copy()

print("\n" + "-" * 65)
print("SAMPLE LINKAGE REVIEW")
print("-" * 65)

print("\nPatients requiring sample linkage review:")

print(len(review))

if not review.empty:

    print(
        review[
            [
                "patientId",
                "gdc_project",
                "published_msi_status",
                "pancancer_gdc_match",
                "published_gdc_match",
                "cbio_sample_match"
            ]
        ]
        .head(15)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 19. Additional cohort validation
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("COHORT VALIDATION CHECKS")
print("-" * 65)

# Check Tier A and Tier B are mutually exclusive.

tier_a_patients = set(
    strict_mss["patientId"]
)

tier_b_patients = set(
    extended_low_msi["patientId"]
)

overlap = tier_a_patients.intersection(
    tier_b_patients
)

if overlap:
    raise RuntimeError(
        "Tier A and Tier B contain overlapping patients."
    )

print("\nTier A / Tier B overlap: 0 patients")

# Check total cohort preservation.

if len(cohort) != len(comparison):
    raise RuntimeError(
        "Patient count changed unexpectedly."
    )

print(
    "Patient count preservation: PASSED"
)

# Check cohort category counts.

if (
    cohort["evidence_tier"]
    .value_counts()
    .sum()
    != len(cohort)
):
    raise RuntimeError(
        "Evidence tier counts do not sum correctly."
    )

print(
    "Evidence tier counts: PASSED"
)

print("\nAll cohort validation checks passed.")

# ------------------------------------------------------------
# 20. Save patient-level outputs
# ------------------------------------------------------------

cohort_path = (
    METADATA_DIR /
    "tcga_mss_evidence_tiers.csv"
)

tier_a_path = (
    METADATA_DIR /
    "tcga_tier_a_published_mss.csv"
)

tier_b_path = (
    METADATA_DIR /
    "tcga_tier_b_computational_low_msi.csv"
)

review_path = (
    METADATA_DIR /
    "tcga_mss_sample_linkage_review.csv"
)

summary_path = (
    RESULTS_DIR /
    "tcga_mss_evidence_tier_summary.csv"
)

cohort.to_csv(
    cohort_path,
    index=False
)

strict_mss.to_csv(
    tier_a_path,
    index=False
)

extended_low_msi.to_csv(
    tier_b_path,
    index=False
)

review.to_csv(
    review_path,
    index=False
)

tier_summary.to_csv(
    summary_path
)

# ------------------------------------------------------------
# 21. Save research methodology
# ------------------------------------------------------------

methodology = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.9",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "sources": [
        "TCGA GDC RNA-seq",
        "cBioPortal PanCancer Atlas 2018",
        "cBioPortal TCGA CRC Published 2012"
    ],

    "tier_a_definition": (
        "Published MSS status with concordant "
        "low MANTIS/MSIsensor scores and "
        "matching sample identifiers."
    ),

    "tier_b_definition": (
        "Concordant low MSI scores without "
        "available published categorical status."
    ),

    "tier_a_patients": int(
        len(strict_mss)
    ),

    "tier_b_patients": int(
        len(extended_low_msi)
    ),

    "sample_barcode_note": (
        "Matching the first 15 characters of "
        "a TCGA barcode establishes patient "
        "and sample-type correspondence, "
        "not identical biospecimen aliquots."
    ),

    "final_expression_samples_selected": False,

    "limitations": [
        "Tier B is not independently verified MSS.",
        "Published MSI-L is excluded from Tier A.",
        "Cross-study samples may represent different aliquots.",
        "Published and computational MSI evidence may not be fully independent.",
        "Selection bias between published and extended cohorts remains possible.",
        "Sample-type barcode matching does not establish identical tissue aliquots."
    ]
}

methodology_path = (
    METADATA_DIR /
    "tcga_mss_evidence_tier_methodology.json"
)

with open(methodology_path, "w") as f:

    json.dump(
        methodology,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 22. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.9 SUMMARY")
print("=" * 65)

print("\nFiles saved:")

print("1. tcga_mss_evidence_tiers.csv")
print("2. tcga_tier_a_published_mss.csv")
print("3. tcga_tier_b_computational_low_msi.csv")
print("4. tcga_mss_sample_linkage_review.csv")
print("5. tcga_mss_evidence_tier_summary.csv")
print("6. tcga_mss_evidence_tier_methodology.json")

print("\nFinal Tier A patients:", len(strict_mss))

print("Final Tier B patients:", len(extended_low_msi))

print("\nNo expression matrices downloaded.")

print("Final RNA-seq sample selection still pending.")

print("\nPATCH 1.9 COMPLETE")
print("=" * 65)